# Master Thesis Pipeline

In [ ]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from master_thesis.config import Config
from master_thesis.constants import CROSSBORDER_PAIRS, NORWAY_ZONES
from master_thesis.data import ExternalMarketDataLoader
from master_thesis.orchestration import run_pipeline

load_dotenv()

In [ ]:
cfg = Config.from_env()
cfg

In [ ]:
pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start": fold.train_start,
            "train_end_exclusive": fold.train_end,
            "validation_start": fold.validation_start,
            "validation_end_exclusive": fold.validation_end,
        }
        for fold in cfg.rolling_folds
    ]
)

In [ ]:
pd.DataFrame(
    {
        "zones": list(NORWAY_ZONES),
        "raw_cache_path": [cfg.raw_cache_path] * len(NORWAY_ZONES),
        "gas_path": [cfg.gas_data_path or cfg.default_gas_path] * len(NORWAY_ZONES),
    }
)

In [ ]:
pd.DataFrame(CROSSBORDER_PAIRS, columns=["from_area", "to_area"])

In [ ]:
external_loader = ExternalMarketDataLoader(cfg)
gas_path = cfg.gas_data_path or cfg.default_gas_path

if not gas_path.exists():
    external_loader.download_gas_prices(gas_path)

gas_prices = external_loader.load_gas_prices()
gas_prices.head()

In [ ]:
run_pipeline(cfg)

In [ ]:
status_path = cfg.output_dir / "run_status.csv"
metrics_path = cfg.output_dir / "metrics.csv"
summary_path = cfg.output_dir / "metrics_validation_summary.csv"
selected_models_path = cfg.output_dir / "selected_models.csv"

run_status = pd.read_csv(status_path)
metrics = pd.read_csv(metrics_path)
validation_summary = pd.read_csv(summary_path)
selected_models = pd.read_csv(selected_models_path)

run_status.head()

In [ ]:
run_status.groupby(["zone", "feature_set", "status"], dropna=False).size().reset_index(name="rows")

In [ ]:
validation_summary.sort_values(["zone", "sample", "mean_rmse"]).head(20)

In [ ]:
selected_models